# ARTICLE2_NB3: F7 (test misspecification), F8 (non-identical baselines) and the Monte Carlo test-size study

**This notebook produces results for the fault-injection study (see README).**

Protocol: *study protocol* (frozen 11 Nov 2026), Section 3 (F7, F8), Section 4 (outcomes) and Section 5 (Monte Carlo).

**What it does**
- **CELL 2 - F7.** On the five reference seeds from NB1: the v1 testing route (plain-variance DM, all 84 pairs, no correction, winner by MAE but tested on squared loss) against the reference route (Newey-West + HLN, 9 primary tests, Holm).
- **CELL 3 - Monte Carlo (RQ4).** How often each testing route names a winner when, by construction, no model is better. 5,000 replications per setting.
- **CELLS 4-5 - Baselines.** The seven literature baselines rebuilt as in v2 (reference) and the v1 route (F8: trained on 2015-2022, recursive 501-step forecasts, same-day indicators), for each seed.
- **CELL 6 - Outcomes.** F7 counts and agreement; Monte Carlo rejection rates with 95% intervals; F8 baseline MAE ratios, materiality and the ranking of architectures against baselines.

**Code:** `03_Code/article2_tests.py` (F7, Monte Carlo) and `03_Code/article2_baselines.py` (baselines, F8). Both import the reference code and never edit it.
**Inputs and outputs:** reads only `Article 2/02_Inputs_v2_snapshot` and `Article 2/05_Outputs/reference`; writes only `Article 2/05_Outputs/tests` and `Article 2/05_Outputs/baselines`.
**Runtime:** about 45-60 minutes on a T4 the first time (the ARIMA grids take about 20 minutes and are cached). Run NB1 Part C first.

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os, sys
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
CODE_DIR = ROOT / "03_Code"
for f in ["article2_pipeline.py", "article2_faults.py", "article2_tests.py", "article2_baselines.py"]:
    assert (CODE_DIR / f).exists(), f"{f} missing from {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
print("Working directory:", ROOT)

## CELL 1: Imports, versions, settings and the reference forecasts from NB1

In [ ]:
import json, time, pickle, importlib, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import sklearn, scipy, statsmodels
warnings.filterwarnings("ignore")

import article2_pipeline as ap
import article2_faults as af
import article2_tests as at
import article2_baselines as ab
importlib.reload(ap); importlib.reload(af); importlib.reload(at); importlib.reload(ab)

P = ap.paths(ROOT)
OUT = P["OUT"]
REF_DIR = OUT / "reference"
TDIR = OUT / "tests"
BDIR = OUT / "baselines"
BCACHE = BDIR / "_cache"
for d in [TDIR, BDIR, BCACHE, OUT / "logs"]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = {"S0": 42, "S1": 101, "S2": 202, "S3": 303, "S4": 404}   # Protocol Section 2 (frozen)
MC_REPS = 5000                                                    # Protocol Section 5
MC_SETTINGS = [(501, 10), (441, 10), (501, 5), (501, 20), (441, 5), (441, 20)]   # main = block 10; 5 and 20 = sensitivity
assert (REF_DIR / "reference_seed_range.csv").exists(), "Run ARTICLE2_NB1 Part C first."

VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "tensorflow": tf.__version__, "xgboost": xgb.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
            "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
for k, v in VERSIONS.items():
    print(f"{k:14s}: {v}")
print("Input snapshot:", ap.check_input_snapshot(P))

data = ap.load_data(P)
arfima_snap = ap.load_arfima(P)
REFC = {s: {t: pd.read_csv(REF_DIR / f"seed_{s}" / f"{t}_test_all_models.csv", parse_dates=["Date"], index_col="Date")
            for t in ap.TICKERS} for s in SEEDS.values()}
print("Reference forecasts loaded for seeds:", list(REFC))

## CELL 2: F7 - the v1 testing route on the reference forecasts (seconds)

For each seed: 84 plain-variance DM tests (8 models, 28 pairs, 3 stocks), no correction, winner named by MAE. Compared with the protocol route: 9 primary tests, Newey-West (5 lags) + HLN, Holm. The same 84 pairs tested the reference way are also shown, so the effect of the variance formula and of the correction can be told apart.

In [ ]:
f7_rows, f7_fams = [], []
for label, seed in SEEDS.items():
    fam = at.f7_family(REFC[seed]).assign(Seed=seed)
    ref9 = ap.primary_dm(REFC[seed])
    f7_fams.append(fam)
    f7_rows.append(at.f7_summary(fam, ref9, seed))
f7_fams = pd.concat(f7_fams, ignore_index=True)
f7_sum = pd.DataFrame(f7_rows)
f7_fams.to_csv(TDIR / "F7_all_84_tests_by_seed.csv", index=False)
f7_sum.to_csv(TDIR / "F7_summary_by_seed.csv", index=False)
print(f7_sum.to_string(index=False))
print()
print("Significant under F7 but not under the reference route (seed S0):")
s0 = f7_fams[(f7_fams.Seed == 42) & f7_fams.Sig_F7 & ~f7_fams.Sig_ref84]
print(s0[["Ticker", "Model_A", "Model_B", "Winner_by_MAE", "Winner_by_squared_loss", "p_plain", "p_Holm84"]].round(4).to_string(index=False))

## CELL 3: Monte Carlo test-size study (about 1-3 minutes)

Null data: the seed S0 reference loss differentials of the 84 pairs, each centred at zero. Whole days are resampled jointly (stationary block bootstrap), so links between pairs and over time are kept. Random seed fixed at 2026 before the run (deviation log).

In [ ]:
mc_file = TDIR / "_mc_raw.pkl"
D0, cols0 = at.loss_differentials(REFC[42])
if mc_file.exists():
    with open(mc_file, "rb") as f:
        mc_raw = pickle.load(f)
else:
    mc_raw = {}
for n, block in MC_SETTINGS:
    if (n, block) in mc_raw:
        continue
    t0 = time.time()
    mc_raw[(n, block)] = at.run_monte_carlo(D0, cols0, n, block, MC_REPS)
    with open(mc_file, "wb") as f:
        pickle.dump(mc_raw, f)
    print(f"n = {n}, mean block {block}: {time.time() - t0:.0f} s")
mc = at.mc_table(mc_raw, MC_REPS)
mc.to_csv(TDIR / "monte_carlo_test_size.csv", index=False)
show = mc.copy()
show["Rate (95% CI)"] = show.apply(lambda r: f"{r.Rate:.3f} ({r.CI95_low_median_test:.3f}-{r.CI95_high_median_test:.3f})", axis=1)
print(show.pivot_table(index="Procedure", columns=["n", "Mean_block"], values="Rate (95% CI)", aggfunc="first").to_string())
print()
print("Single tests: the interval is for the median of the 84 tests; the full range over the 84 tests is in the CSV.")

## CELL 4: ARIMA and ARFIMA for the baselines (about 20-25 minutes the first time, CPU; cached)

- Reference: ARIMA AIC grid (p, q 0-5; d 0-1) on 2015-2020, one-step forecasts (v2).
- F8: the same grid on 2015-2022 with one static 501-step forecast; the v2 ARFIMA grid re-estimated on 2015-2022 with one static 501-step forecast (for ARFIMA -> LSTM).

In [ ]:
bd = ab.baseline_data(data, arfima_snap)
afile = BCACHE / "arima_arfima.pkl"
if afile.exists():
    with open(afile, "rb") as f:
        arima_ref, arima8, arfima8, arima_logs = pickle.load(f)
    print("Loaded from cache")
else:
    t0 = time.time()
    arima_ref, l1 = ab.arima_reference(bd)
    arima8, l2 = ab.arima_f8(bd)
    arfima8, l3 = ab.arfima_f8(data)
    arima_logs = pd.concat([l1, l2, l3], ignore_index=True)
    with open(afile, "wb") as f:
        pickle.dump((arima_ref, arima8, arfima8, arima_logs), f)
    print(f"Estimated in {time.time() - t0:.0f} s")
arima_logs.to_csv(BDIR / "arima_arfima_selection.csv", index=False)
print(arima_logs.to_string(index=False))

## CELL 5: Baselines for every seed - reference and F8 (about 3-5 minutes per seed on a T4; cached)

Seed S0 = 42 is checked against the v2 baseline results (v2). A small difference can come from a different GPU; it is recorded, not hidden, and F8 is always compared with the reference re-run in this session.

In [ ]:
B = {}
for label, seed in SEEDS.items():
    cfile = BCACHE / f"seed_{seed}.pkl"
    if cfile.exists():
        with open(cfile, "rb") as f:
            B[seed] = pickle.load(f)
        print(f"{label} (seed {seed}) loaded from cache")
        continue
    t0 = time.time()
    ref_tabs, ref_log = ab.run_reference_baselines(bd, arima_ref, seed)
    f8_tabs, f8_log = ab.run_f8_baselines(bd, arima8, arfima8, seed)
    B[seed] = {"ref": ref_tabs, "f8": f8_tabs, "log": pd.concat([ref_log, f8_log], ignore_index=True)}
    with open(cfile, "wb") as f:
        pickle.dump(B[seed], f)
    print(f"{label} (seed {seed}) done in {time.time() - t0:.0f} s")

for seed in SEEDS.values():
    for route in ["ref", "f8"]:
        for t in ap.TICKERS:
            B[seed][route][t].to_csv(BDIR / f"{route}_seed_{seed}_{t}_baselines.csv")
pd.concat([B[s]["log"] for s in SEEDS.values()]).to_csv(BDIR / "baseline_training_logs.csv", index=False)

m42 = ab.baseline_metrics(B[42]["ref"], "REF", 42).pivot(index="Model", columns="Ticker", values="MAE").loc[ab.BASELINES]
bdiff = (m42 - ab.V2_BASELINE_MAE).abs()
VERDICT_BASE = "MATCHES_4DP" if (bdiff <= 0.00005 + 1e-12).all().all() else "ENVIRONMENT_DRIFT"
print()
print("Seed 42 reference baselines vs v2 (test MAE, USD): largest difference", round(float(bdiff.max().max()), 5))
print(bdiff.round(5).to_string())
print("Baseline reproduction verdict:", VERDICT_BASE)

## CELL 6: Outcomes

- **F7:** significant results under the v1 route versus the reference route, and agreement on the 9 primary tests.
- **Monte Carlo:** rejection rates when no model is truly better (nominal 5% for single tests).
- **F8:** Delta MAE ratio of each baseline (F8 minus same-seed reference); material if outside the reference seed range (three decimals) or Kendall's tau of the full ranking (research models, baselines, persistence) is below 1.

In [ ]:
pd.set_option("display.width", 200)
print("F7 - mean over seeds [min, max]")
for c in ["F7_significant_of_84", "Ref_way_significant_of_84", "Ref_protocol_significant_of_9",
          "F7_significant_of_9_primary", "Primary_agreement_of_9", "Direction_conflicts_of_84",
          "F7_significant_with_direction_conflict"]:
    print(f"  {c:40s} {f7_sum[c].mean():6.1f} [{f7_sum[c].min()}, {f7_sum[c].max()}]")
F7_MATERIAL = bool((f7_sum.F7_significant_of_84 != f7_sum.Ref_protocol_significant_of_9).any())
print("  F7 material (significance count changes):", F7_MATERIAL)

print()
print("MONTE CARLO - main setting (mean block 10)")
main = mc[mc.Mean_block == 10].pivot_table(index="Procedure", columns="n", values="Rate")
print(main.round(3).to_string())

# F8 outcomes
bm = []
for seed in SEEDS.values():
    bm.append(ab.baseline_metrics(B[seed]["ref"], "REF", seed))
    bm.append(ab.baseline_metrics(B[seed]["f8"], "F8", seed))
    bm.append(ab.research_metrics(REFC[seed], seed))
bm = pd.concat(bm, ignore_index=True)
bm.to_csv(BDIR / "baseline_and_research_metrics_long.csv", index=False)

def with_mean(df):
    m = df.groupby(["Run", "Seed", "Model", "Type"])[["MAE_ratio"]].mean().reset_index().assign(Ticker="Mean")
    return pd.concat([df, m], ignore_index=True)

bmm = with_mean(bm)
ref_b = bmm[bmm.Run == "REF"]
f8_b = bmm[bmm.Run == "F8"]
rng_b = ref_b.groupby(["Model", "Ticker"])["MAE_ratio"].agg(Ref_min="min", Ref_max="max").reset_index()
d8 = f8_b.merge(ref_b[["Seed", "Model", "Ticker", "MAE_ratio"]], on=["Seed", "Model", "Ticker"], suffixes=("", "_ref"))
d8["Delta"] = d8.MAE_ratio - d8.MAE_ratio_ref
s8 = d8.groupby(["Model", "Ticker"]).agg(F8_ratio=("MAE_ratio", "mean"), Ref_ratio=("MAE_ratio_ref", "mean"),
                                         Delta_mean=("Delta", "mean"), Delta_min=("Delta", "min"),
                                         Delta_max=("Delta", "max")).reset_index().merge(rng_b, on=["Model", "Ticker"])
s8["Outside_ref_range"] = (s8.F8_ratio.round(3) < s8.Ref_min.round(3)) | (s8.F8_ratio.round(3) > s8.Ref_max.round(3))
s8.to_csv(BDIR / "F8_delta_summary.csv", index=False)

def ranking(run_rows, research_rows):
    r = pd.concat([run_rows, research_rows]).groupby("Model")["MAE_ratio"].mean()
    r["Persistence"] = 1.0
    return r

rank_rows, ref_noise = [], {}
for seed in SEEDS.values():
    res = bm[(bm.Run == "research") & (bm.Seed == seed)]
    r_ref = ranking(bm[(bm.Run == "REF") & (bm.Seed == seed)], res)
    r_f8 = ranking(bm[(bm.Run == "F8") & (bm.Seed == seed)], res).loc[r_ref.index]
    tau = float(ap.stats.kendalltau(r_ref.round(3).values, r_f8.round(3).values).statistic)
    best_arch = r_ref[["Serial (Final)", "Parallel Equal", "Parallel InvError", "Parallel SoftMax", "Stacking"]].min()
    rank_rows.append({"Seed": seed, "Tau": tau,
                      "Baselines_better_than_best_architecture_REF": int((r_ref[ab.BASELINES] < best_arch).sum()),
                      "Baselines_better_than_best_architecture_F8": int((r_f8[ab.BASELINES] < best_arch).sum()),
                      "Baselines_better_than_persistence_REF": int((r_ref[ab.BASELINES] < 1).sum()),
                      "Baselines_better_than_persistence_F8": int((r_f8[ab.BASELINES] < 1).sum())})
    if seed != 42:
        r0 = ranking(bm[(bm.Run == "REF") & (bm.Seed == 42)], bm[(bm.Run == "research") & (bm.Seed == 42)]).loc[r_ref.index]
        ref_noise[seed] = float(ap.stats.kendalltau(r0.round(3).values, r_ref.round(3).values).statistic)
rank8 = pd.DataFrame(rank_rows)
rank8.to_csv(BDIR / "F8_ranking_by_seed.csv", index=False)
F8_MATERIAL = bool(s8.loc[s8.Ticker == "Mean", "Outside_ref_range"].any() or (rank8.Tau < 1 - 1e-12).any())

print()
print("F8 - baseline MAE ratio, stock mean (mean over seeds): reference -> F8, Delta [min, max]")
t8 = s8[s8.Ticker == "Mean"].set_index("Model").loc[ab.BASELINES]
for m, r in t8.iterrows():
    print(f"  {m:20s} {r.Ref_ratio:.3f} -> {r.F8_ratio:.3f}   {r.Delta_mean:+.3f} [{r.Delta_min:+.3f}, {r.Delta_max:+.3f}]"
          f"{'   outside ref range' if r.Outside_ref_range else ''}")
print()
print("F8 - per stock (mean over seeds):")
print(s8[s8.Ticker != "Mean"].pivot(index="Model", columns="Ticker", values="F8_ratio").loc[ab.BASELINES].round(3).to_string())
print()
print("F8 - ranking against the research models:")
print(rank8.round(3).to_string(index=False))
print("Seed noise alone, tau of reference seeds vs S0:", {k: round(v, 3) for k, v in ref_noise.items()})
print("F8 material:", F8_MATERIAL)

## CELL 7: CHECKPOINT and run log

In [ ]:
checks = {
    "f7_five_seeds": f7_sum.Seed.nunique() == 5 and bool((f7_fams.groupby("Seed").size() == 84).all()),
    "monte_carlo_all_settings": all(s in mc_raw for s in MC_SETTINGS),
    "baselines_five_seeds_both_routes": all(len(B[s]["ref"]) == 3 and len(B[s]["f8"]) == 3 for s in SEEDS.values()),
    "baseline_seed42_matches_v2": VERDICT_BASE == "MATCHES_4DP",
}
print(pd.Series(checks).to_string())
log = {"notebook": "ARTICLE2_NB3_Tests_Baselines_MonteCarlo", "purpose": "Article 2 only", "root": str(ROOT),
       "run_at": datetime.now().isoformat(timespec="seconds"), "versions": VERSIONS, "seeds": SEEDS,
       "mc_reps": MC_REPS, "mc_seed": at.MC_SEED, "mc_settings": MC_SETTINGS,
       "baseline_verdict": VERDICT_BASE, "f7_material": F7_MATERIAL, "f8_material": F8_MATERIAL, "checks": checks}
with open(OUT / "logs" / "ARTICLE2_NB3_run_log.json", "w") as f:
    json.dump(log, f, indent=2, default=str)
soft = [k for k in ["baseline_seed42_matches_v2"] if not checks[k]]
hard = [k for k, v in checks.items() if not v and k not in soft]
if soft:
    print("Needs a deviation-log entry (not a stop):", soft)
assert not hard, f"CHECKPOINT FAILED: {hard}"
print()
print("Hard checks: ALL PASSED")
print("Please share the outputs of CELLS 2, 3, 4, 5, 6 and 7.")